# LROC NAC Downloader (by location)

Downloads LROC (Lunar Reconnaissance Orbiter Camera) **Narrow Angle Camera** `.IMG` files
from the WashU Lunar Orbital Data Explorer (ODE) REST API, filtered by a lat/lon bounding box.

**Run the cells in order.** Only the parameters in the last cell need editing for normal use.

> Longitude is **0-360° East** on this site. If you have a negative longitude, add 360
> (e.g. -30 → 330) before entering it below.

## 1. Setup

In [ ]:
!pip install -q requests

import os
import time
import shutil
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from typing import Optional

import requests

ODE_REST_BASE = "https://oderest.rsl.wustl.edu/live2/"
PAGE_SIZE = 100
DEFAULT_TARGET = "moon"
DEFAULT_IHID = "LRO"
DEFAULT_IID = "LROC"
DEFAULT_PT = "EDRNAC4"  # raw NAC EDR (PDS4)


@dataclass(frozen=True)
class BoundingBox:
    minlat: float
    maxlat: float
    westlon: float
    eastlon: float

## 2. Pure functions — talking to the ODE REST API

In [ ]:
def ode_get(session: requests.Session, params: dict, retries: int = 3, timeout: int = 60) -> dict:
    """GET the ODE REST API and return the parsed JSON body, retrying on transient errors."""
    url = ODE_REST_BASE + "?" + urllib.parse.urlencode(params)
    last_err: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:  # noqa: BLE001
            last_err = e
            time.sleep(1.5 * attempt)
    raise RuntimeError(f"ODE REST request failed after {retries} attempts: {last_err}\nURL: {url}")


def bbox_params(bbox: BoundingBox, ihid: str, iid: str, pt: str, target: str) -> dict:
    """Build the base query-string params for a location search."""
    return {
        "target": target,
        "ihid": ihid,
        "iid": iid,
        "pt": pt,
        "query": "product",
        "results": "fmp",  # files + metadata + product info
        "output": "JSON",
        "minlat": bbox.minlat,
        "maxlat": bbox.maxlat,
        "westlon": bbox.westlon,
        "eastlon": bbox.eastlon,
    }


def get_product_count(session: requests.Session, bbox: BoundingBox,
                       ihid: str = DEFAULT_IHID, iid: str = DEFAULT_IID,
                       pt: str = DEFAULT_PT, target: str = DEFAULT_TARGET) -> int:
    """Return the total number of matching products for a bounding box (no download)."""
    params = dict(bbox_params(bbox, ihid, iid, pt, target), results="c")
    result = ode_get(session, params)["ODEResults"]
    if result.get("Status") != "Success":
        raise RuntimeError(f"ODE REST error: {result.get('Error', result)}")
    return int(result.get("Count", 0))


def _page_of_products(session: requests.Session, bbox: BoundingBox, offset: int,
                       ihid: str, iid: str, pt: str, target: str) -> list:
    """Fetch a single page (<=100) of raw product dicts."""
    params = dict(bbox_params(bbox, ihid, iid, pt, target), offset=offset)
    result = ode_get(session, params)["ODEResults"]
    if result.get("Status") != "Success":
        raise RuntimeError(f"ODE REST error: {result.get('Error', result)}")
    products = result.get("Products")
    if not isinstance(products, dict):  # e.g. the string "No Products Found"
        return []
    products = products.get("Product", [])
    return [products] if isinstance(products, dict) else products


def fetch_products(session: requests.Session, bbox: BoundingBox, limit: Optional[int] = None,
                    ihid: str = DEFAULT_IHID, iid: str = DEFAULT_IID,
                    pt: str = DEFAULT_PT, target: str = DEFAULT_TARGET) -> list:
    """Fetch up to `limit` unique product dicts matching the bounding box (all, if limit is None)."""
    total = get_product_count(session, bbox, ihid, iid, pt, target)
    print(f"[info] ODE reports {total} matching product(s) for this location.")

    seen_ids: set = set()
    products: list = []
    offset = 0
    while offset < total:
        page = _page_of_products(session, bbox, offset, ihid, iid, pt, target)
        if not page:
            break
        for p in page:
            pid = p.get("pdsid")
            if pid in seen_ids:
                continue
            seen_ids.add(pid)
            products.append(p)
            if limit and len(products) >= limit:
                return products
        offset += PAGE_SIZE
    return products

## 3. Pure functions — turning products into download jobs

In [ ]:
def extract_img_file(product: dict) -> Optional[dict]:
    """Return the Product_file entry for the .IMG data file, or None if absent."""
    files = product.get("Product_files", {}).get("Product_file", [])
    if isinstance(files, dict):
        files = [files]
    for f in files:
        name = (f.get("FileName") or "").upper()
        if f.get("Type") == "Product" and name.endswith(".IMG"):
            return f
    return None


def build_jobs(products: list) -> list:
    """Map product dicts -> list of (product_id, img_file_dict), skipping products with no .IMG."""
    jobs = []
    for p in products:
        img = extract_img_file(p)
        pid = p.get("pdsid", "unknown")
        if img is None:
            print(f"[warn] {pid}: no .IMG file found, skipping.")
            continue
        jobs.append((pid, img))
    return jobs

## 4. Pure(ish) functions — downloading

In [ ]:
def download_one(session: requests.Session, pid: str, img: dict, outdir: str,
                  overwrite: bool = False, chunk_size: int = 1 << 20) -> tuple:
    """Download a single .IMG file. Returns (product_id, dest_path, status)."""
    dest = os.path.join(outdir, img.get("FileName") or f"{pid}.IMG")
    if not overwrite and os.path.exists(dest) and os.path.getsize(dest) > 0:
        return pid, dest, "skipped (already downloaded)"
    tmp = dest + ".part"
    try:
        with session.get(img["URL"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(tmp, "wb") as fh:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        fh.write(chunk)
        os.replace(tmp, dest)
        return pid, dest, "downloaded"
    except Exception as e:  # noqa: BLE001
        return pid, dest, f"FAILED: {e}"


def download_all(session: requests.Session, jobs: list, outdir: str,
                  workers: int = 4, overwrite: bool = False) -> list:
    """Download every (pid, img) job in parallel. Returns a list of (pid, dest, status)."""
    os.makedirs(outdir, exist_ok=True)
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(download_one, session, pid, img, outdir, overwrite) for pid, img in jobs]
        for fut in as_completed(futures):
            pid, dest, status = fut.result()
            print(f"[{status}] {pid} -> {dest}")
            results.append((pid, dest, status))
    return results

## 5. Orchestrator

Single entry point that composes the functions above: search by location → cap by limit → download.

In [ ]:
def download_lroc_nac_by_location(minlat: float, maxlat: float, westlon: float, eastlon: float,
                                   limit: Optional[int] = None, outdir: str = "/content/nac_downloads",
                                   workers: int = 4, overwrite: bool = False,
                                   pt: str = DEFAULT_PT) -> list:
    """
    Search LROC NAC products in a lat/lon box and download their .IMG files.

    Args:
        minlat, maxlat: latitude range.
        westlon, eastlon: longitude range, 0-360 East.
        limit: max number of products to download (None = all matches).
        outdir: local folder to save .IMG files into.
        workers: parallel download threads.
        overwrite: re-download even if a file already exists.
        pt: ODE product type, EDRNAC4 (raw, default) or CDRNAC4 (calibrated).

    Returns:
        List of (product_id, local_path, status) tuples.
    """
    bbox = BoundingBox(minlat=minlat, maxlat=maxlat, westlon=westlon, eastlon=eastlon)
    session = requests.Session()
    session.headers.update({"User-Agent": "lroc-nac-colab/1.0"})

    products = fetch_products(session, bbox, limit=limit, pt=pt)
    jobs = build_jobs(products)
    print(f"[info] {len(jobs)} .IMG file(s) queued for download (limit={limit}).")

    return download_all(session, jobs, outdir, workers=workers, overwrite=overwrite)

## 6. Run it

Edit the parameters below and run this cell.

In [ ]:
results = download_lroc_nac_by_location(
    minlat=25,
    maxlat=25.2,
    westlon=30,     # 0-360 East! negative lon? add 360, e.g. -30 -> 330
    eastlon=30.2,
    limit=5,        # how many products to download; set to None for all matches
    outdir="/content/nac_downloads",
    workers=4,
)

ok = sum(1 for _, _, status in results if status in ("downloaded", "skipped (already downloaded)"))
print(f"\n[done] {ok}/{len(results)} file(s) present on disk.")

## 7. (Optional) Zip and download the files to your computer

In [ ]:
from google.colab import files

zip_path = shutil.make_archive("/content/nac_downloads", "zip", "/content/nac_downloads")
files.download(zip_path)